# Grid-distance comparison

Two open proxies stand in for "is there a network nearby": the LINZ powerline
layer and road centrelines. This notebook asks how much the screening answer
depends on which one you believe, and shows why S-06 is published as two
distances plus a verify flag rather than folded into a score.

**Scope warning.** The committed spatial outputs are deterministic DEMO
geometries with eight surviving fixtures, not Canterbury parcels. Every overlap
number below illustrates the method on n = 8. It is not an empirical estimate
of proxy disagreement, and the notebook prints the sample size beside the
statistic so that it cannot be quoted as one.

In [ ]:
import sys
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))

from nz_solar_siting.grid_distance import compare_top_n
from nz_solar_siting.siting import SitingConfig

candidates = gpd.read_file(ROOT / "outputs" / "demo" / "candidates.gpkg")
print(f"candidates: {len(candidates)} DEMO features (not parcels)")
candidates[["site_id", "grid_line_m", "road_proxy_m", "grid_rank", "road_rank", "rank_shift"]]

## How fast does the shortlist change with N?

A single top-N Jaccard number hides how unstable the comparison is. Sweeping N
shows that the two proxies agree least at the sharp end and converge only as
the shortlist approaches the whole candidate set - which is another way of
saying the ranking carries little information here.

In [ ]:
sweep = pd.DataFrame([compare_top_n(candidates, n) for n in range(2, len(candidates) + 1)])
sweep["sample_note"] = f"demo n={len(candidates)}"
sweep[["n", "overlap_count", "non_overlap_count", "jaccard", "grid_only", "road_only"]]

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.8))
ax.plot(sweep["n"], sweep["jaccard"], marker="o", color="#2d6a9f")
ax.set(
    xlabel="Shortlist size N",
    ylabel="Jaccard overlap",
    ylim=(0, 1.05),
    title=f"Top-N agreement between the two proxies (DEMO, n={len(candidates)})",
)
ax.grid(alpha=0.2)
fig.tight_layout()

## Which sites would the verify flag catch?

`S06_verify_grid` fires when either distance exceeds `grid_distance_review_m`
or the two ranks disagree by at least `rank_shift_review`. Both thresholds live
in `config/assumptions.yml`; the cell below re-derives the flag from the
published distances so the rule can be checked without re-running the pipeline.

In [ ]:
config = SitingConfig()
recomputed = (
    candidates[["grid_line_m", "road_proxy_m"]].max(axis=1) > config.grid_distance_review_m
) | (candidates["rank_shift"] >= config.rank_shift_review)
check = candidates[["site_id", "grid_line_m", "road_proxy_m", "rank_shift", "S06_verify_grid"]].copy()
check["recomputed_flag"] = recomputed
assert check["recomputed_flag"].equals(check["S06_verify_grid"].astype(bool))
print(f"verify-grid flags: {int(recomputed.sum())} of {len(check)}")
check

## Why grid distance is not in `screen_score`

If the two proxies disagree this much, a weighted score built on them would
hand a reviewer a confident-looking ordering derived from the least reliable
input. `screen_score` therefore uses only solar resource and usable area, and
grid proximity is carried as two published distances plus a flag. Moving the
powerline layer changes the distances and leaves the ordering untouched.

In [ ]:
from nz_solar_siting.demo_data import build_demo_layers
from nz_solar_siting.siting import evaluate_sites

sites, conservation, powerlines, roads = build_demo_layers()
baseline, _ = evaluate_sites(sites, conservation, powerlines, roads)
moved = powerlines.copy()
moved["geometry"] = moved.geometry.translate(xoff=50_000.0)
shifted, _ = evaluate_sites(sites, conservation, moved, roads)

comparison = pd.DataFrame({
    "site_id": baseline["site_id"],
    "grid_line_m": baseline["grid_line_m"],
    "grid_line_m_after_move": shifted["grid_line_m"],
    "screen_score": baseline["screen_score"],
    "screen_score_after_move": shifted["screen_score"],
})
print("score unchanged:", comparison["screen_score"].equals(comparison["screen_score_after_move"]))
comparison

## The same comparison on real geometry

Everything above is a method demonstration on twelve rectangles. LINZ Topo50 and
LCDB need portal accounts, so the repository also ships an OpenStreetMap
substitute for the Canterbury plains. The rules that OSM can support - S-01 area
and S-02 width - leave a sample in the thousands, which is enough for the
comparison to mean something.

OSM is not LINZ: completeness varies by area and contributor, and a mapped
land-use polygon is a land-use observation, not a parcel title. Read what
follows as an OSM measurement.

In [ ]:
import json

from nz_solar_siting.osm_layers import read_osm_layers, split_by_voltage

import yaml

study_config = yaml.safe_load(
    (ROOT / "config" / "assumptions.yml").read_text(encoding="utf-8")
)["osm_study"]
connection_tier = study_config["connection_tier"]

farmland, powerlines, roads, wetland, coastline = read_osm_layers(ROOT / "data" / "derived" / "osm")
study = json.loads((ROOT / "outputs" / "osm" / "osm_grid_study.json").read_text(encoding="utf-8"))
print(f"farmland polygons downloaded : {study['farmland_polygons_downloaded']:,}")
print(f"passing area and width       : {study['sites_passing_area_and_width']:,}")
print(f"connection tier              : {connection_tier}")
study["voltage_tier_features"]

## Split by voltage, not by OSM tag

The `power` tag does not separate the network the way a connection decision
does. The extract has 66 kV ways tagged `line` and 66 kV ways tagged
`minor_line`, and it lumps 220 kV transmission and 350 kV HVDC in with the
first group. A Canterbury project of tens of megawatts connects at 33 or 66 kV.
Ways carrying two circuits, such as `66000;11000`, take the higher one.

In [ ]:
voltage = powerlines["voltage"].map(lambda label: label if isinstance(label, str) else "untagged")
crosstab = pd.crosstab(powerlines["power"], voltage)
shared = [column for column in crosstab.columns if ";" in column]
print("ways carrying more than one circuit:", int(crosstab[shared].to_numpy().sum()))
crosstab[[c for c in ("11000", "33000", "66000", "220000", "350000", "untagged") if c in crosstab]]

In [ ]:
tiers, counts = split_by_voltage(
    powerlines, study_config["voltage_tiers"], float(study_config["excluded_voltage_v"])
)
summary = pd.DataFrame({
    "mapped_ways": {name: len(frame) for name, frame in tiers.items()},
    "median_distance_m": {
        name: study["median_distance_m"][f"{name}_m"] for name in tiers
    },
    "p90_distance_m": {name: study["distance_p90_m"][f"{name}_m"] for name in tiers},
    "rank_correlation_with_road": {
        name: study["rank_correlation"][f"{name}_m|road_m"] for name in tiers
    },
})
summary.loc["road"] = [
    study["road_features"], study["median_distance_m"]["road_m"],
    study["distance_p90_m"]["road_m"], 1.0,
]
summary

The ordering is monotonic in voltage, and that is the finding. Roads follow the
low-voltage poles, so they predict 11 kV proximity reasonably well. They say
almost nothing about the 33-66 kV tier a project would actually connect to.

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4))
for pair, sweep in study["top_n_sweep"].items():
    left, right = pair.split("|")
    if right != "road_m":
        continue
    frame = pd.DataFrame(sweep)
    ax.plot(frame["n"], frame["jaccard"], marker="o", label=left.removesuffix("_m"))
ax.set(
    xlabel="Shortlist size N",
    ylabel="Jaccard overlap with the road shortlist",
    ylim=(0, 1.02),
    title=f"How much a road proxy recovers, by voltage tier (n = {study['sites_passing_area_and_width']:,})",
)
ax.legend(frameon=False, fontsize=9)
ax.grid(alpha=0.2)
fig.tight_layout()

## A flag that fires on everything is not a flag

The first version of this study flagged a site when *any* proxy distance
exceeded 5 km, which flags almost every site here. The flag now tests distance
to the connection tier against that tier's own review distance. The absolute
rank-shift trigger was dropped: `rank_shift_review = 3` was chosen against eight
demo fixtures, and at this sample size the median shift is in the hundreds, so
the same constant fires everywhere. Rank thresholds expressed in ranks do not
survive a change of sample size.

In [ ]:
osm = pd.read_csv(ROOT / "outputs" / "osm" / "osm_grid_distance.csv")
tier_columns = [f"{name}_m" for name in study_config["voltage_tiers"]]
old_style = (osm[tier_columns + ["road_m"]].max(axis=1) > 5000) | (
    osm[f"{connection_tier}_road_rank_shift"] >= 3
)
comparison = pd.DataFrame({
    "flagged": [int(old_style.sum()), int(osm["S06_verify_grid"].sum())],
    "share": [
        round(old_style.mean(), 3),
        round(osm["S06_verify_grid"].mean(), 3),
    ],
}, index=["max over every proxy, or rank shift >= 3", "connection-tier distance only"])
comparison

## The aerial step is not optional

All twenty of the largest connection-tier-versus-road disagreements were checked
against LINZ Basemaps imagery. Each verdict cites the image it was written from,
under `data/aerial/`, and an `observed_detail` naming something visible in that
image, so a claim can be checked rather than taken on trust. The verdicts were
made by an AI agent in the repository author's session and have not been
independently confirmed by a person; the log says so in its `reviewer` column.

**Read the rate carefully.** These twenty were selected *because* they are the
largest disagreements, which biases them towards coast and peninsula edges. The
rate below describes the road proxy's worst cases. It is not the false-positive
rate of the candidate population.

In [ ]:
queue = pd.read_csv(ROOT / "outputs" / "osm" / "aerial_review_queue.csv")
print(f"reviewed {queue['finding'].notna().sum()} of {len(queue)}")
print(f"not developable: {(queue['developable'] == 'no').sum()} "
      f"({study['aerial_review']['false_positive_rate']:.0%} of this disagreement-selected sample)")
queue.groupby("developable")[
    [f"{connection_tier}_m", "road_m", "mean_slope_deg", "coastline_m", "area_ha"]
].median().round(1)

In [ ]:
for _, row in queue[queue["developable"] == "no"].iterrows():
    print(f"{row.site_id}  {row.area_ha:>6.1f} ha  slope {row.mean_slope_deg:>5.1f} deg  "
          f"coast {row.coastline_m:>7,.0f} m  water {row.water_m:>7,.0f} m")
    print(f"  seen: {row.observed_detail}")
    print(f"  {row.finding}\n")

## Closing the gap the review exposed

None of those eleven failures is a grid problem. They are steep, wet or coastal,
and the baseline had no rule for any of it - S-08 slope was documented as "not
implemented". Three rules now exist: mean slope from the Copernicus GLO-30 DEM,
intersection with mapped water or wetland, and a coastal proximity flag.

Exclusion and flagging are kept apart below, because a flagged site still
reaches a human as a candidate. The thresholds were chosen with these twenty
labels in view, so what follows is
**in-sample**. It shows the rules express what the imagery showed. It is not an
accuracy claim on unseen sites.

In [ ]:
check = study["aerial_review"]["in_sample_rule_check"]
terrain = study["terrain_water"]
pd.Series({
    "labelled not developable": (queue["developable"] == "no").sum(),
    "excluded by slope": check["excluded_by_slope"],
    "excluded by mapped water": check["excluded_by_water"],
    "excluded, total": check["excluded_total"],
    "flagged only by coast (still a candidate)": check["flagged_only_by_coast"],
    "neither excluded nor flagged": len(check["neither_excluded_nor_flagged"]),
    "developable sites wrongly excluded": check["developable_sites_wrongly_excluded"],
}).to_frame("sites")

In [ ]:
print("neither excluded nor flagged:", check["neither_excluded_nor_flagged"])
missed = queue[queue["site_id"].isin(check["neither_excluded_nor_flagged"])]
missed[["site_id", "area_ha", "mean_slope_deg", "coastline_m", "water_m", "finding"]]

The miss is instructive: it is a flat barrier-spit polygon whose nearest mapped
coastline is 1.3 km away, because the barrier is wide at that point. A simple
setback cannot reach it. Catching it needs either a barrier-landform test or
real land-cover data, and inventing a threshold that happens to capture this one
site would be fitting the sample rather than screening.

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.2))
osm = pd.read_csv(ROOT / "outputs" / "osm" / "osm_grid_distance.csv")
ax.scatter(osm["coastline_m"].clip(lower=10), osm["mean_slope_deg"].clip(lower=0.1),
           s=6, alpha=0.25, color="#6d7f8b", label=f"all {len(osm):,} sites")
labels = queue.merge(osm[["site_id"]], on="site_id")
for verdict, colour in (("yes", "#19a974"), ("no", "#d64545")):
    subset = queue[queue["developable"] == verdict]
    ax.scatter(subset["coastline_m"].clip(lower=10), subset["mean_slope_deg"].clip(lower=0.1),
               s=70, color=colour, edgecolor="#101820", label=f"reviewed: {verdict}")
ax.axhline(terrain["maximum_mean_slope_deg"], linestyle="--", color="#101820", linewidth=1)
ax.axvline(terrain["coastal_review_distance_m"], linestyle=":", color="#101820", linewidth=1)
ax.set(xscale="log", yscale="log", xlabel="Distance to mapped coastline (m)",
       ylabel="Mean slope (degrees)", title="Where the reviewed sites sit against the new rules")
ax.legend(frameon=False, fontsize=8)
ax.grid(alpha=0.2, which="both")
fig.tight_layout()

In [ ]:
pd.Series({
    "study population": study["sites_passing_area_and_width"],
    "excluded by slope": terrain["excluded_by_slope"],
    "excluded by mapped water": terrain["excluded_by_water"],
    "excluded by either": terrain["excluded_by_either"],
    "flagged for coastal review": terrain["flagged_coastal"],
    "surviving": terrain["surviving_sites"],
}).to_frame("sites")

## What is still missing

The LINZ Topo50 and LCDB versions of this same run, and an independent human
check of the twenty aerial verdicts. Both are blocked on things outside the
code. Nothing here should be quoted as a LINZ result.

Attribution: (c) OpenStreetMap contributors, ODbL 1.0. Aerial imagery (c) LINZ
and Environment Canterbury, CC BY 4.0. Elevation from Copernicus GLO-30 DEM,
(c) European Union / ESA.